In [1]:
import pm4py
import json
import copy
from collections import defaultdict

INPUT_PATH = "commitizen_role_annotated.json"

ocel = pm4py.read_ocel2_json(INPUT_PATH)

with open(INPUT_PATH) as f:
    data = json.load(f)

print("Events:", len(ocel.events))
print("Objects:", len(ocel.objects))
print("Object types:", ocel.objects[ocel.object_type_column].value_counts().to_dict())

Events: 21488
Objects: 6534
Object types: {'commit': 2765, 'task': 1721, 'issue': 1459, 'user': 589}


In [2]:
# Build role -> set of user object IDs mapping from the annotated JSON
role_to_users = defaultdict(set)

for obj in data["objects"]:
    if obj["type"] != "user":
        continue
    role = next(
        (a["value"] for a in obj.get("attributes", []) if a["name"] == "role"),
        None
    )
    if role:
        role_to_users[role].add(obj["id"])

for role, users in sorted(role_to_users.items(), key=lambda x: -len(x[1])):
    print(f"{role:25s}  {len(users)} users")

issue_reporter             425 users
contributor                66 users
quality_engineer           37 users
technical_writer           23 users
feature_developer          20 users
maintainer                 8 users
bot                        6 users
devops_engineer            4 users


In [3]:
# For each role: collect events where at least one user of that role participates,
# then include all objects linked to those events.

def filter_ocel_by_user_ids(data: dict, user_ids: set) -> dict:
    """
    Return a new OCEL dict containing only:
    - events that involve at least one user in `user_ids`
    - all objects referenced by those events
    """
    # Event IDs that involve at least one target user
    target_event_ids = {
        event["id"]
        for event in data["events"]
        for rel in event.get("relationships", [])
        if rel["objectId"] in user_ids
    }

    # Collect all object IDs referenced by those events
    referenced_object_ids = {
        rel["objectId"]
        for event in data["events"]
        if event["id"] in target_event_ids
        for rel in event.get("relationships", [])
    }

    filtered = copy.deepcopy(data)
    filtered["events"] = [
        e for e in filtered["events"] if e["id"] in target_event_ids
    ]
    filtered["objects"] = [
        o for o in filtered["objects"] if o["id"] in referenced_object_ids
    ]
    return filtered


role_logs = {}
for role, user_ids in role_to_users.items():
    role_logs[role] = filter_ocel_by_user_ids(data, user_ids)

print("Filtered logs created for roles:", list(role_logs.keys()))

Filtered logs created for roles: ['issue_reporter', 'bot', 'feature_developer', 'quality_engineer', 'contributor', 'maintainer', 'technical_writer', 'devops_engineer']


In [4]:
# Summary table: events and objects per role log
import pandas as pd

rows = []
for role, log in sorted(role_logs.items()):
    obj_type_counts = defaultdict(int)
    for obj in log["objects"]:
        obj_type_counts[obj["type"]] += 1
    rows.append({
        "role": role,
        "events": len(log["events"]),
        **{f"objects_{t}": obj_type_counts.get(t, 0)
           for t in ["user", "commit", "task", "issue"]},
    })

df_summary = pd.DataFrame(rows).set_index("role")
df_summary

,events,objects_user,objects_commit,objects_task,objects_issue
role,,,,,
bot,3394,10,0,0,1045
contributor,236,66,236,84,109
devops_engineer,15,4,15,13,7
feature_developer,100,20,100,65,27
issue_reporter,16037,427,0,0,1436
maintainer,2149,8,2149,1392,837
quality_engineer,194,37,194,104,60
technical_writer,71,23,71,63,37


In [5]:
# Save each role log to its own file
import os

OUTPUT_DIR = "role_logs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

saved = []
for role, log in role_logs.items():
    path = os.path.join(OUTPUT_DIR, f"ocel_{role}.json")
    with open(path, "w") as f:
        json.dump(log, f)
    saved.append(path)
    print(f"Saved {path}  ({len(log['events'])} events, {len(log['objects'])} objects)")

Saved role_logs/ocel_issue_reporter.json  (16037 events, 1863 objects)
Saved role_logs/ocel_bot.json  (3394 events, 1055 objects)
Saved role_logs/ocel_feature_developer.json  (100 events, 212 objects)
Saved role_logs/ocel_quality_engineer.json  (194 events, 395 objects)
Saved role_logs/ocel_contributor.json  (236 events, 495 objects)
Saved role_logs/ocel_maintainer.json  (2149 events, 4386 objects)
Saved role_logs/ocel_technical_writer.json  (71 events, 194 objects)
Saved role_logs/ocel_devops_engineer.json  (15 events, 39 objects)


In [6]:
# Verify: reload each log with pm4py and confirm event counts match
for path in saved:
    ocel_role = pm4py.read_ocel2_json(path)
    role_name = os.path.splitext(os.path.basename(path))[0].replace("ocel_", "")
    print(f"{role_name:25s}  events={len(ocel_role.events):5d}  objects={len(ocel_role.objects):5d}")

issue_reporter             events=16037  objects= 1863
bot                        events= 3394  objects= 1055
feature_developer          events=  100  objects=  212
quality_engineer           events=  194  objects=  395
contributor                events=  236  objects=  495
maintainer                 events= 2149  objects= 4386
technical_writer           events=   71  objects=  194
devops_engineer            events=   15  objects=   39
